**Task 2 – Word-Level Language Model (LSTM)**

This task implements a word-level language model trained on *Poirot Investigates* by Agatha Christie.  
The objective is to predict the next word in a sequence and generate coherent text in a similar literary style.

The model is implemented using a two-layer LSTM architecture in PyTorch.

**1. Data Loading**

The dataset is the Project Gutenberg version of
"Poirot Investigates". The text file is loaded from Google Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = "/content/drive/MyDrive/Colab Notebooks/61262-0.txt"

In [4]:
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])

*** START OF THE PROJECT GUTENBERG EBOOK 61262 ***

  POIROT INVESTIGATES




  BY THE SAME AUTHOR

  THE MYSTERIOUS AFFAIR AT STYLES

  THE SECRET ADVERSARY

  THE MURDER ON THE LINKS

  THE BODLEY HEAD




  POIROT INVESTIGATES

  BY AGATHA CHRISTIE




  LONDON

  JOHN LANE THE BODLEY HEAD LIMITED




  First published in Great Britain by
  John Lane Company, The Bodley Head Limited, 1924

  Copyright © 1924 Agatha Christie Limited




  CONTENTS


  I The Adventure of “The Western Star”

  I


**2. Data Preprocessing**

The following preprocessing steps were applied:

- Removed Project Gutenberg header and footer
- Converted text to lowercase
- Removed unnecessary special characters
- Tokenised text at word level

In [5]:
import re

def preprocess_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    start = text.find("POIROT INVESTIGATES")
    end = text.find("END OF THE PROJECT GUTENBERG")
    text = text[start:end]

    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s.,!?]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text

text = preprocess_text(file_path)
words = text.split()

print("Total words:", len(words))
print("First 50 words:", words[:50])

Total words: 52543
First 50 words: ['poirot', 'investigates', 'by', 'the', 'same', 'author', 'the', 'mysterious', 'affair', 'at', 'styles', 'the', 'secret', 'adversary', 'the', 'murder', 'on', 'the', 'links', 'the', 'bodley', 'head', 'poirot', 'investigates', 'by', 'agatha', 'christie', 'london', 'john', 'lane', 'the', 'bodley', 'head', 'limited', 'first', 'published', 'in', 'great', 'britain', 'by', 'john', 'lane', 'company,', 'the', 'bodley', 'head', 'limited,', '1924', 'copyright', '1924']


**3. Vocabulary Construction**

A word-level vocabulary was constructed from the dataset.
Each unique word is mapped to a numerical index.

In [6]:
import torch
import numpy as np

# Build vocabulary
vocab = sorted(set(words))
vocab_size = len(vocab)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("Vocabulary size:", vocab_size)

Vocabulary size: 9048


**4. Sequence Preparation**

Training samples were created using a sliding window approach.

- Sequence length = 20 words
- Target = next word

A sequence length of 20 was chosen as a balance between computational efficiency and contextual learning.  
Shorter sequences limit context understanding, while longer sequences significantly increase memory requirements.

In [7]:
sequence_length = 20

inputs = []
targets = []

for i in range(len(words) - sequence_length):
    seq = words[i:i+sequence_length]
    target = words[i+sequence_length]

    inputs.append([word2idx[w] for w in seq])
    targets.append(word2idx[target])

X = torch.tensor(inputs)
y = torch.tensor(targets)

print("Input shape:", X.shape)
print("Target shape:", y.shape)

Input shape: torch.Size([52523, 20])
Target shape: torch.Size([52523])


**5. DataLoader**

The dataset was batched using PyTorch DataLoader for efficient training.

In [12]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

**6. LSTM Model Architecture**

The model consists of:

- Embedding layer (128 dimensions)
- Two LSTM layers (hidden size = 256)
- Dropout (0.2) to reduce overfitting
- Fully connected output layer

LSTM was chosen over a Transformer due to the relatively small dataset size (~52k words).
Transformers typically require much larger datasets to generalise effectively.


In [13]:
import torch.nn as nn

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256, num_layers=2):
        super(LSTMLanguageModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=0.2)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out[:, -1, :])  # last timestep
        return out, hidden

    def init_hidden(self, batch_size, device):
        return (torch.zeros(2, batch_size, 256).to(device),
                torch.zeros(2, batch_size, 256).to(device))

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


**7. Model Training**

Hyperparameters:

- Embedding size: 128
- Hidden size: 256
- Layers: 2
- Learning rate: 0.001
- Optimizer: Adam
- Loss function: CrossEntropyLoss
- Epochs: 20

The training loss decreased from 7.10 to 1.25, indicating successful learning.
However, the low final loss suggests potential memorisation due to limited dataset size.

In [16]:
model = LSTMLanguageModel(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in dataloader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        hidden = model.init_hidden(batch_X.size(0), device)

        optimizer.zero_grad()
        output, hidden = model(batch_X, hidden)
        loss = criterion(output, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1/20, Loss: 7.1150
Epoch 2/20, Loss: 6.5686
Epoch 3/20, Loss: 6.1899
Epoch 4/20, Loss: 5.8651
Epoch 5/20, Loss: 5.5408
Epoch 6/20, Loss: 5.2016
Epoch 7/20, Loss: 4.8433
Epoch 8/20, Loss: 4.4679
Epoch 9/20, Loss: 4.0886
Epoch 10/20, Loss: 3.7180
Epoch 11/20, Loss: 3.3636
Epoch 12/20, Loss: 3.0395
Epoch 13/20, Loss: 2.7423
Epoch 14/20, Loss: 2.4793
Epoch 15/20, Loss: 2.2490
Epoch 16/20, Loss: 2.0336
Epoch 17/20, Loss: 1.8520
Epoch 18/20, Loss: 1.6765
Epoch 19/20, Loss: 1.5225
Epoch 20/20, Loss: 1.3795


**8. Saving Model Weights**


In [17]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/Colab Notebooks/lstm_model.pth")

torch.save(word2idx,
           "/content/drive/MyDrive/Colab Notebooks/word2idx.pth")

torch.save(idx2word,
           "/content/drive/MyDrive/Colab Notebooks/idx2word.pth")

print("Model saved successfully!")

Model saved successfully!


In [18]:
def generate_text(model, start_text, length=80, temperature=0.8):
    model.eval()

    words_input = start_text.lower().split()
    generated = words_input.copy()

    hidden = model.init_hidden(1, device)

    for _ in range(length):
        input_seq = generated[-sequence_length:]

        if len(input_seq) < sequence_length:
            input_seq = ["poirot"] * (sequence_length - len(input_seq)) + input_seq

        input_idx = [word2idx.get(w, 0) for w in input_seq]
        input_tensor = torch.tensor([input_idx]).to(device)

        output, hidden = model(input_tensor, hidden)

        probs = torch.softmax(output / temperature, dim=1).detach().cpu().numpy()[0]
        next_word_idx = np.random.choice(len(probs), p=probs)

        next_word = idx2word[next_word_idx]
        generated.append(next_word)

    return " ".join(generated)

In [19]:
prompt = "poirot stood by the window"
print(generate_text(model, prompt, 80))

poirot stood by the window it destroyed the first timethe black first quite up, and uncomprehending. the prime ministers car had never noticed him that impression of course, poirot remained forward out of the darkness. she was a changed woman. a flaw. passed. he had never been sitting out of the window to examine him for some time. but so we say, now for some of the fact that he has shaved the duplicate japp! of him. but no so face now has a bad


In [20]:
prompt = "poirot stood by the window"
print(generate_text(model, prompt, 80, temperature=0.5))


poirot stood by the window of romance. he gave dinner to my tuesday, again, by the papers. i saw at once that he was going to stroll down to the village and post back at once. she knew one with a peculiar excitement. as he ceased to struggle poirot got back the count, please, behind him the tale of his card, and then we heard the second man failed. ascanio, yes, neatly to be found in the premises. i had been going through with him


**9. Text Generation**

The model generates text using temperature sampling.
A minimum of 20 words is generated as required.

In [21]:
prompt = input("Enter starting words: ")
print(generate_text(model, prompt, 80, temperature=0.8))

Enter starting words: HI WHERE ARE YOU '
hi where are you ' removed and turned to him that he had slippers arrived in england. he soothes himself about him to call which when she spoke to each time in a state of manner. poirot introduced him of his victims and returned with his notebook. he was clearly endeavouring to bear him a fortnight incredible! what does the great workmen from the station, of its being required. taking up to charing cross, by the district, a matter of captain cottage, where is his
